Inventario y estructura de archivos

In [1]:
import pandas as pd
import os

# === 1. Ruta base ===
ruta_campos = r"C:\Users\Vìctor\OneDrive\Desktop\ESP_Project\Datos"

# === 2. Cargar inventario de archivos Excel ===
files = [f for f in os.listdir(ruta_campos) if f.endswith(".xlsx")]

print(f"✅ {len(files)} archivos encontrados.\n")

# === 3. Explorar cada archivo Excel ===
for file in files:
    archivo_path = os.path.join(ruta_campos, file)
    try:
        # Listar las hojas (worksheets) dentro del archivo
        xls = pd.ExcelFile(archivo_path)
        print(f"📘 {file} -> contiene {len(xls.sheet_names)} hojas:")
        print("   Hojas:", ", ".join(xls.sheet_names[:5]), 
              "..." if len(xls.sheet_names) > 5 else "")
        
        # Cargar la primera hoja como muestra
        df = pd.read_excel(xls, xls.sheet_names[0])
        print("   ▪️ Filas:", len(df), "| Columnas:", len(df.columns))
        print("   ▪️ Columnas principales:", list(df.columns[:6]), "\n")
        
    except Exception as e:
        print(f"⚠️ Error al leer {file}: {e}\n")

print("✅ Exploración de estructura completada.")



✅ 2 archivos encontrados.

📘 CampoSachamMejoresPozos.xlsx -> contiene 1 hojas:
   Hojas: MejoresPozos 
   ▪️ Filas: 47 | Columnas: 31
   ▪️ Columnas principales: ['POZO', 'Unnamed: 1', 'FECHA', 'CAMPO', 'Unnamed: 4', 'P. CAS'] 

📘 Datos_por_pozo.xlsx -> contiene 48 hojas:
   Hojas: POTENCIAL, SCHAO-473HS1UI, SCH-128HS, SCHA-409HI, SCHAO-477HUI ...
   ▪️ Filas: 128 | Columnas: 21
   ▪️ Columnas principales: ['FECHA:', 'Junio/3/2018', 'Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4', 'Unnamed: 5'] 

✅ Exploración de estructura completada.


Obtener la lista de pozos (nombre de cada hoja)

In [10]:
import pandas as pd

# Ruta del archivo Excel con todos los pozos
ruta_excel = r"C:\Users\Vìctor\OneDrive\Desktop\ESP_Project\Datos\Datos_por_pozo.xlsx"

# Cargar el archivo y listar todas las hojas
xls = pd.ExcelFile(ruta_excel)
pozos = [s for s in xls.sheet_names if s.upper() != "POTENCIAL"]

listado_pozos=[]
# Mostrar la lista
print("✅ Pozos encontrados:")
for p in pozos:
    
    listado_pozos.append(p)

print(f"\nTotal de pozos: {len(pozos)}")
print(listado_pozos)


✅ Pozos encontrados:

Total de pozos: 47
['SCHAO-473HS1UI', 'SCH-128HS', 'SCHA-409HI', 'SCHAO-477HUI', 'SCH-106HS', 'SCHV-226HS', 'SCH-124UI', 'SCH-186UI', 'SCHAC-353HI', 'SCHH-218HS', 'SCHAH-404HI', 'SCHP-181UI', 'SCH-050UI', 'SCHH-239HS', 'SCHAE-373HUI', 'SCH-127HS', 'SCH-123UI', 'SCHV-227HS', 'SCHAM-452HI', 'SCH-083HI', 'SCHAR-503UI', 'SCHAE-375UI', 'SCHAO-471R1TI', 'SCH-007U', 'SCHA-419HI', 'SCHP-188UI', 'SCHAF-539HS1UI', 'SCHQ-468UI', 'SCHAF-529HUI', 'SCH-018UI', 'SCHAR-500UI', 'SCH-134U', 'SCHT-229UI', 'SCHA-418UI', 'SCHAQ-497UI', 'SCHAB-614UI', 'SCHAQ-492UI', 'SCH-087BT', 'SCHH-237R1HS', 'SCH-088T', 'SCHAQ-490UI', 'SCHAO-478HUI', 'SCHAO-475HUI', 'SCH-142UI', 'SCHAB-316UI', 'SCHAJ-428TI', 'FORECAST']


# === Construir rutas con os.path

In [14]:
import pandas as pd
import os

# === 1. Rutas base ===
base_dir = r"C:\Users\Vìctor\OneDrive\Desktop\ESP_Project\Datos"
ruta_campos = os.path.join(base_dir, "CampoSachamMejoresPozos.xlsx")
ruta_datos = os.path.join(base_dir, "Datos_por_pozo.xlsx")

# === 2. Cargar tabla de POZOS y CAMPOS ===
df_campos = pd.read_excel(ruta_campos, sheet_name="MejoresPozos")
df_campos = df_campos.rename(columns=lambda x: x.strip())
df_campos = df_campos[['POZO', 'CAMPO']].dropna()
df_campos['POZO'] = df_campos['POZO'].str.upper().str.strip()
df_campos['CAMPO'] = df_campos['CAMPO'].str.upper().str.strip()

from openpyxl import load_workbook

def contar_columnas_excel(ruta, hoja):
    try:
        wb = load_workbook(ruta, read_only=True)
        ws = wb[hoja]
        return ws.max_column
    except Exception as e:
        print(f"❌ No se pudo contar columnas en hoja {hoja}: {e}")
        return 0
for hoja in hojas_pozos:
    try:
        n_cols = contar_columnas_excel(ruta_datos, hoja)
        if n_cols == 0:
            continue

        letra_final = chr(64 + n_cols) if n_cols <= 26 else 'Z'  # límite A-Z
        df_pozo = pd.read_excel(ruta_datos,
                                sheet_name=hoja,
                                usecols=f"A:{letra_final}",
                                engine="openpyxl")

        if df_pozo.empty:
            print(f"⚠️ Hoja vacía: {hoja}")
            continue

        df_pozo['POZO'] = hoja.upper().strip()
        df_total = pd.concat([df_total, df_pozo], ignore_index=True)

    except Exception as e:
        print(f"❌ Error en hoja {hoja}: {e}")


Hacer el merge con la tabla de campos

In [52]:
# === 1. Normalizar nombres de pozo (por si acaso)
df_total['POZO'] = df_total['POZO'].str.upper().str.strip()

# === 2. Hacer el merge con la tabla de campos
df_final = df_total.merge(df_campos, on='POZO', how='left')

# === 3. Manejar columnas duplicadas si aparecen CAMPO_x y CAMPO_y
if 'CAMPO_y' in df_final.columns:
    df_final['CAMPO'] = df_final['CAMPO_y']
    df_final.drop(columns=['CAMPO_x', 'CAMPO_y'], inplace=True)

# === 4. Verificar pozos sin campo asignado
faltantes = df_final[df_final['CAMPO'].isnull()]['POZO'].unique()

if len(faltantes) > 0:
    print(f"⚠️ Pozos sin campo asignado ({len(faltantes)}):")
    for p in faltantes:
        print(f"   - {p}")
else:
    print("✅ Todos los pozos tienen campo asignado.")

# === 5. Vista previa
print("\n📋 Vista previa con campo integrado:")
display(df_final.head())


✅ Todos los pozos tienen campo asignado.

📋 Vista previa con campo integrado:


,@Name( ),Date,PRUEBA DE PRODUCCION PETRÓLEO A 24 HORAS bbl/d,PRUEBA DE PRODUCCIÓN AGUA A 24 HORAS bbl/d,Prueba_pozo.Oil_24 + Prueba_pozo.Water_24,PRUEBA DE PRODUCCIÓN GAS A 24 HORAS Mcf/d,BSW %,GRAVEDAD API DEL PETROLEO,SALINIDAD PPM PU,PRESION DE INTAKE psi,FRECUENCIA BOMBA Hz,AMPERAJE BOMBA Amp,PRESION DE TUBING psi,PRESION DE CASING psi,TIPO DE BOMBA,TEMPERATURA DE LA BOMBA Deg. F,ETAPAS DE LA BOMBA,POZO,CAMPO,CAMPO
0,SCHAO-473HS1UI,2015-02-01 00:00:00,1436.0,359.0,1795.0,0.0,20.00,18.2,NaN,NaN,NaN,NaN,170.0,NaN,NaN,NaN,NaN,SCHAO-473HS1UI,NaN,SACHA NORTE 1
1,SCHAO-473HS1UI,2015-10-01 00:00:00,1544.0,386.0,1930.0,0.0,20.00,18.2,NaN,NaN,NaN,NaN,170.0,NaN,NaN,NaN,NaN,SCHAO-473HS1UI,NaN,SACHA NORTE 1
2,SCHAO-473HS1UI,2/18/2015,1714.0,429.0,2143.0,0.0,20.02,18.2,NaN,NaN,NaN,NaN,150.0,NaN,NaN,NaN,NaN,SCHAO-473HS1UI,NaN,SACHA NORTE 1
3,SCHAO-473HS1UI,2/23/2015,1737.0,434.0,2171.0,0.0,19.99,18.2,NaN,NaN,NaN,NaN,140.0,NaN,NaN,NaN,NaN,SCHAO-473HS1UI,NaN,SACHA NORTE 1
4,SCHAO-473HS1UI,2/25/2015,1650.0,521.0,2171.0,0.0,24.00,18.2,NaN,NaN,NaN,NaN,140.0,NaN,NaN,NaN,NaN,SCHAO-473HS1UI,NaN,SACHA NORTE 1


#Ultimas filas

In [53]:
display(df_final.tail())

,@Name( ),Date,PRUEBA DE PRODUCCION PETRÓLEO A 24 HORAS bbl/d,PRUEBA DE PRODUCCIÓN AGUA A 24 HORAS bbl/d,Prueba_pozo.Oil_24 + Prueba_pozo.Water_24,PRUEBA DE PRODUCCIÓN GAS A 24 HORAS Mcf/d,BSW %,GRAVEDAD API DEL PETROLEO,SALINIDAD PPM PU,PRESION DE INTAKE psi,FRECUENCIA BOMBA Hz,AMPERAJE BOMBA Amp,PRESION DE TUBING psi,PRESION DE CASING psi,TIPO DE BOMBA,TEMPERATURA DE LA BOMBA Deg. F,ETAPAS DE LA BOMBA,POZO,CAMPO,CAMPO
25029,SCHAJ-428TI,2025-11-09 00:00:00,208.0,831.0,1039.0,60.26,79.98,28.5,44800.0,323.0,55.0,34.0,240.0,50.0,SF-900|SF-900|SF-900|SF-900,280.0,492.0,SCHAJ-428TI,NaN,SACHA SUR
25030,SCHAJ-428TI,9/18/2025,209.0,834.0,1043.0,60.40,79.96,28.5,44800.0,310.0,56.0,36.0,260.0,50.0,SF-900|SF-900|SF-900|SF-900,274.0,492.0,SCHAJ-428TI,NaN,SACHA SUR
25031,SCHAJ-428TI,9/20/2025,205.0,822.0,1027.0,59.25,80.04,28.5,44800.0,308.0,56.0,36.0,300.0,50.0,SF-900|SF-900|SF-900|SF-900,275.0,492.0,SCHAJ-428TI,NaN,SACHA SUR
25032,SCHAJ-428TI,2025-02-10 00:00:00,208.0,830.0,1038.0,59.25,79.96,28.5,44800.0,300.0,57.0,37.0,300.0,50.0,SF-900|SF-900|SF-900|SF-900,279.0,492.0,SCHAJ-428TI,NaN,SACHA SUR
25033,SCHAJ-428TI,10/22/2025,208.0,830.0,1038.0,59.25,79.96,28.5,44800.0,300.0,57.0,37.0,300.0,50.0,SF-900|SF-900|SF-900|SF-900,279.0,492.0,SCHAJ-428TI,NaN,SACHA SUR


Eliminar columnas nulas y repetidas

In [54]:
df_final.drop(df_final.columns[-2], axis=1, inplace=True)
print("✅ Columnas actuales:")
print(df_final.columns.tolist())
df_final




✅ Columnas actuales:
['    @Name( )  ', '      Date ', 'PRUEBA DE PRODUCCION PETRÓLEO A 24 HORAS bbl/d ', 'PRUEBA DE PRODUCCIÓN AGUA A 24 HORAS bbl/d ', '    Prueba_pozo.Oil_24 + Prueba_pozo.Water_24  ', 'PRUEBA DE PRODUCCIÓN GAS A 24 HORAS Mcf/d ', '  BSW   % ', 'GRAVEDAD API DEL PETROLEO  ', 'SALINIDAD   PPM PU ', 'PRESION DE INTAKE psi ', 'FRECUENCIA BOMBA Hz  ', 'AMPERAJE BOMBA Amp  ', 'PRESION DE TUBING psi ', 'PRESION DE CASING psi ', 'TIPO DE BOMBA  ', 'TEMPERATURA DE LA BOMBA Deg. F ', 'ETAPAS DE LA BOMBA  ', 'POZO', 'CAMPO']


,@Name( ),Date,PRUEBA DE PRODUCCION PETRÓLEO A 24 HORAS bbl/d,PRUEBA DE PRODUCCIÓN AGUA A 24 HORAS bbl/d,Prueba_pozo.Oil_24 + Prueba_pozo.Water_24,PRUEBA DE PRODUCCIÓN GAS A 24 HORAS Mcf/d,BSW %,GRAVEDAD API DEL PETROLEO,SALINIDAD PPM PU,PRESION DE INTAKE psi,FRECUENCIA BOMBA Hz,AMPERAJE BOMBA Amp,PRESION DE TUBING psi,PRESION DE CASING psi,TIPO DE BOMBA,TEMPERATURA DE LA BOMBA Deg. F,ETAPAS DE LA BOMBA,POZO,CAMPO
0,SCHAO-473HS1UI,2015-02-01 00:00:00,1436.0,359.0,1795.0,0.00,20.00,18.2,NaN,NaN,NaN,NaN,170.0,NaN,NaN,NaN,NaN,SCHAO-473HS1UI,SACHA NORTE 1
1,SCHAO-473HS1UI,2015-10-01 00:00:00,1544.0,386.0,1930.0,0.00,20.00,18.2,NaN,NaN,NaN,NaN,170.0,NaN,NaN,NaN,NaN,SCHAO-473HS1UI,SACHA NORTE 1
2,SCHAO-473HS1UI,2/18/2015,1714.0,429.0,2143.0,0.00,20.02,18.2,NaN,NaN,NaN,NaN,150.0,NaN,NaN,NaN,NaN,SCHAO-473HS1UI,SACHA NORTE 1
3,SCHAO-473HS1UI,2/23/2015,1737.0,434.0,2171.0,0.00,19.99,18.2,NaN,NaN,NaN,NaN,140.0,NaN,NaN,NaN,NaN,SCHAO-473HS1UI,SACHA NORTE 1
4,SCHAO-473HS1UI,2/25/2015,1650.0,521.0,2171.0,0.00,24.00,18.2,NaN,NaN,NaN,NaN,140.0,NaN,NaN,NaN,NaN,SCHAO-473HS1UI,SACHA NORTE 1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
25029,SCHAJ-428TI,2025-11-09 00:00:00,208.0,831.0,1039.0,60.26,79.98,28.5,44800.0,323.0,55.0,34.0,240.0,50.0,SF-900|SF-900|SF-900|SF-900,280.0,492.0,SCHAJ-428TI,SACHA SUR
25030,SCHAJ-428TI,9/18/2025,209.0,834.0,1043.0,60.40,79.96,28.5,44800.0,310.0,56.0,36.0,260.0,50.0,SF-900|SF-900|SF-900|SF-900,274.0,492.0,SCHAJ-428TI,SACHA SUR
25031,SCHAJ-428TI,9/20/2025,205.0,822.0,1027.0,59.25,80.04,28.5,44800.0,308.0,56.0,36.0,300.0,50.0,SF-900|SF-900|SF-900|SF-900,275.0,492.0,SCHAJ-428TI,SACHA SUR
25032,SCHAJ-428TI,2025-02-10 00:00:00,208.0,830.0,1038.0,59.25,79.96,28.5,44800.0,300.0,57.0,37.0,300.0,50.0,SF-900|SF-900|SF-900|SF-900,279.0,492.0,SCHAJ-428TI,SACHA SUR


In [ ]:
import os

# Ruta base
base_dir = r"C:\Users\Vìctor\OneDrive\Desktop\ESP_Project\Datos"

# Nombre del nuevo archivo Excel
nombre_archivo = "df_final_integrado.xlsx"

# Ruta completa del archivo de salida
ruta_salida = os.path.join(base_dir, nombre_archivo)

# Guardar DataFrame en formato Excel
df_final.to_excel(ruta_salida, index=False)

print(f"Archivo guardado exitosamente en:\n{ruta_salida}")


✅ Archivo guardado exitosamente en:
C:\Users\Vìctor\OneDrive\Desktop\ESP_Project\Datos\df_final_integrado.xlsx
